# Embryo AI Pipeline Verification

This notebook verifies the end-to-end inference flow of the Embryo AI system, ensuring that edge cases, invalid inputs, and valid inputs are processed correctly across all three models (Validator, Classifier, Grader).

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import traceback

# Import custom modules (ensure we are in the root directory where these exist)
from predict_validator import EmbryoValidator
from predict_module import EmbryoClassifier
from predict_malpani import EmbryoPredictor

In [ ]:
print("Initializing Pipeline Models...")

try:
    validator = EmbryoValidator(model_path="embryo_validator_model.h5")
    classifier = EmbryoClassifier(model_path="embryo_model_turbo.h5")
    grader = EmbryoPredictor(model_path="embryo_grading_v4.pth")
    print("Models successfully initialized.")
except Exception as e:
    print(f"Warning during initialization: {e}")
    # Fallback instances will be created inside run_pipeline if needed, or modules will handle missing models gracefully.

## 1. Pipeline Definition

In [ ]:
def run_pipeline(image_path, show_image=True):
    """
    Executes the full Embryo AI diagnostic pipeline on a single image.
    Validator (Stage 0) -> Stage Classification (Phase 1) -> Grading (Phase 2, if Blastocyst)
    """
    results = {
        'status': 'Processing',
        'is_valid': None,
        'val_confidence': None,
        'stage': None,
        'stage_confidence': None,
        'grading': None,
        'human_review_required': False,
        'error': None
    }
    
    if show_image and os.path.exists(image_path):
        try:
            img = Image.open(image_path)
            plt.figure(figsize=(3, 3))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
        except:
            pass
            
    try:
        # 1. Validation
        is_valid, val_conf = validator.validate(image_path)
        results['is_valid'] = is_valid
        results['val_confidence'] = val_conf
        
        if not is_valid:
            results['status'] = 'Rejected by Validator'
            return results
            
        # 2. Stage Classification
        stage, stage_conf, _ = classifier.predict(image_path)
        results['stage'] = stage
        results['stage_confidence'] = stage_conf
        
        if stage_conf < 0.70:
            results['human_review_required'] = True
            
        # 3. Grading (Only if Blastocyst)
        if stage == "Blastocyst":
            full_grade, grading_res = grader.predict(image_path)
            results['grading'] = grading_res
            if grading_res and grading_res.get('low_confidence', False):
                 results['human_review_required'] = True
                 
        results['status'] = 'Success'
        
    except Exception as e:
        results['status'] = 'Error'
        results['error'] = str(e)
        
    return results

def print_results(res):
    print("--- Pipeline Results ---")
    for k, v in res.items():
        if v is not None:
            print(f"{k.capitalize().replace('_', ' ')}: {v}")
    print("------------------------\n")

## 2. Test Case Implementations

In [ ]:
import cv2

# Helper: Generate synthetic test cases
os.makedirs("test_cases", exist_ok=True)

# Generate Blank Image
blank_img = np.zeros((160, 160, 3), dtype=np.uint8)
cv2.imwrite("test_cases/blank.jpg", blank_img)

# Generate Corrupted Image (Create a file with random text data but jpg extension)
with open("test_cases/corrupted.jpg", "w") as f:
    f.write("This is not a real image file, just text bytes simulating corruption.")

# Generate Extremely Noisy Image
noisy_img = np.random.randint(0, 256, (160, 160, 3), dtype=np.uint8)
cv2.imwrite("test_cases/noisy.jpg", noisy_img)

print("Synthetic test cases generated.")

In [ ]:
# Test 1: Invalid Image (Random Photo / Logo)
print("\n===============================")
print("TEST 1: Invalid Image (Logo)")
res1 = run_pipeline("embryo_ai_logo_1776785134392.png")
print_results(res1)

In [ ]:
# Test 2: Valid Embryo (Assuming 2b.jpeg is an early stage or blastocyst)
print("\n===============================")
print("TEST 2: Valid Embryo (2b.jpeg)")
res2 = run_pipeline("2b.jpeg")
print_results(res2)

In [ ]:
# Test 3: Valid Blastocyst
# (If 2b.jpeg is not a blastocyst, this test behaves as Test 2. 
# For true blastocyst test, a valid blastocyst image path is needed).
print("\n===============================")
print("TEST 3: Blastocyst Pipeline Check")
res3 = run_pipeline("2b.jpeg") # Use a specific blastocyst image if available
if res3['stage'] == 'Blastocyst':
    print("Blastocyst detected, grading executed.")
else:
    print("Not a blastocyst. Grading skipped successfully.")
print_results(res3)

In [ ]:
# Test 4: Edge Cases (Blank, Corrupted, Noisy)
print("\n===============================")
print("TEST 4A: Blank Image")
res_blank = run_pipeline("test_cases/blank.jpg", show_image=False)
print_results(res_blank)

print("\n===============================")
print("TEST 4B: Corrupted File")
res_corrupt = run_pipeline("test_cases/corrupted.jpg", show_image=False)
print_results(res_corrupt)

print("\n===============================")
print("TEST 4C: Extremely Noisy Image")
res_noisy = run_pipeline("test_cases/noisy.jpg", show_image=False)
print_results(res_noisy)